In [ ]:
# ============================================================
# Cell 1. Install packages
# ============================================================

%pip install -q pandas numpy requests tqdm python-dotenv openpyxl lxml beautifulsoup4 statsmodels scipy

In [ ]:
# ============================================================
# Cell 2. Configuration
# ============================================================

from pathlib import Path
from getpass import getpass
from calendar import monthrange
import os
import json
import time
import zipfile
import re
import io
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from dotenv import load_dotenv, set_key

import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project folders
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CACHE_DIR = DATA_DIR / "cache"
OUT_DIR = PROJECT_DIR / "output_final_clean_representativeness"

for d in [DATA_DIR, RAW_DIR, CACHE_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Input file
# ------------------------------------------------------------

KRX_FILE = Path(r"C:\Users\starw\Downloads\krx_market_list_clean.csv")

if not KRX_FILE.exists():
    raise FileNotFoundError(f"KRX file not found: {KRX_FILE}")

# ------------------------------------------------------------
# Analysis settings
# ------------------------------------------------------------

START_FISCAL_YEAR = 2018
END_FISCAL_YEAR = 2024
FISCAL_YEARS = list(range(START_FISCAL_YEAR, END_FISCAL_YEAR + 1))

# 사업보고서는 보통 다음 해 3월에 제출되므로,
# fiscal_year 2018 사업보고서는 filing_year 2019에 주로 나타남.
FILING_YEARS_FOR_UNIVERSE = list(range(START_FISCAL_YEAR + 1, END_FISCAL_YEAR + 2))

REPORT_CODE = "11011"

PILOT_MODE = False
PILOT_N_FIRMS = 50

REQUEST_SLEEP = 0.20
TIMEOUT = 40
MAX_RETRIES = 3

# 최종 메인 분석표본
# "current_sample" 권장: 현재 KRX 상장 KOSPI 비금융기업 기준
# "annual_dart_universe"는 robustness용
ANALYSIS_SAMPLE_MODE = "current_sample"

# ------------------------------------------------------------
# Open DART API key
# ------------------------------------------------------------

ENV_PATH = PROJECT_DIR / ".env"
load_dotenv(ENV_PATH)

DART_API_KEY = os.getenv("DART_API_KEY")

if not DART_API_KEY:
    DART_API_KEY = getpass("Open DART API key를 입력하세요: ").strip()
    set_key(str(ENV_PATH), "DART_API_KEY", DART_API_KEY)
    print(f"API key saved to {ENV_PATH}")
else:
    print("DART_API_KEY loaded from .env")

BASE_URL = "https://opendart.fss.or.kr/api"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT_DIR:", OUT_DIR)
print("KRX_FILE:", KRX_FILE)
print("FISCAL_YEARS:", FISCAL_YEARS)
print("FILING_YEARS_FOR_UNIVERSE:", FILING_YEARS_FOR_UNIVERSE)
print("ANALYSIS_SAMPLE_MODE:", ANALYSIS_SAMPLE_MODE)

In [ ]:
# ============================================================
# Cell 3. Utility functions
# ============================================================

def clean_stock_code(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.0$", "", s)
    s = re.sub(r"[^0-9]", "", s)
    if s == "":
        return np.nan
    return s.zfill(6)


def clean_corp_code(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.0$", "", s)
    s = re.sub(r"[^0-9]", "", s)
    if s == "":
        return np.nan
    return s.zfill(8)


def is_standard_stock_code(x):
    return bool(re.fullmatch(r"\d{6}", str(x).strip()))


def parse_number(x):
    if pd.isna(x):
        return np.nan
    
    s = str(x).strip()
    
    if s in ["", "-", "—", "–", "nan", "None", "해당사항 없음", "해당사항없음"]:
        return np.nan
    
    if "해당" in s and "없" in s:
        return np.nan
    
    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s[1:-1]
    
    s = s.replace(",", "")
    matches = re.findall(r"-?\d+\.?\d*", s)
    
    if not matches:
        return np.nan
    
    val = float(matches[0])
    if neg:
        val = -abs(val)
    return val


def safe_divide(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return a / b


def winsorize_series(s, lower=0.01, upper=0.99):
    s = s.copy()
    if s.dropna().empty:
        return s
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def cache_path(endpoint_name, params):
    key_parts = [endpoint_name]
    for k in sorted(params.keys()):
        if k == "crtfc_key":
            continue
        key_parts.append(f"{k}-{params[k]}")
    fname = "__".join(key_parts)
    fname = re.sub(r"[^A-Za-z0-9가-힣_\-\.]", "_", fname)
    return CACHE_DIR / f"{fname}.json"


def dart_get_json(endpoint_name, params, use_cache=True):
    params = dict(params)
    params["crtfc_key"] = DART_API_KEY
    
    cp = cache_path(endpoint_name, params)
    if use_cache and cp.exists():
        with open(cp, "r", encoding="utf-8") as f:
            return json.load(f)
    
    url = f"{BASE_URL}/{endpoint_name}"
    last_error = None
    
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            time.sleep(REQUEST_SLEEP)
            r = requests.get(url, params=params, timeout=TIMEOUT)
            r.raise_for_status()
            data = r.json()
            
            if use_cache:
                with open(cp, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
            return data
        
        except Exception as e:
            last_error = str(e)
            time.sleep(attempt)
    
    return {"status": "ERROR", "message": last_error, "list": []}


def dart_list(endpoint_name, params, use_cache=True):
    data = dart_get_json(endpoint_name, params, use_cache=use_cache)
    status = str(data.get("status", ""))
    
    if status == "000":
        return data.get("list", [])
    elif status == "013":
        return []
    else:
        print(f"[WARN] {endpoint_name} status={status}, message={data.get('message')}")
        return []


def save_df(df, filename):
    path = OUT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved: {path} | shape={df.shape}")
    return path


def print_basic(df, name, n=5):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head(n))


def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def month_ranges(year):
    ranges = []
    for m in range(1, 13):
        start = f"{year}{m:02d}01"
        end = f"{year}{m:02d}{monthrange(year, m)[1]:02d}"
        ranges.append((start, end))
    return ranges

In [ ]:
# ============================================================
# Cell 4. Load KRX and build current-listed KOSPI nonfinancial sample
# ============================================================

krx_raw = pd.read_csv(KRX_FILE, dtype=str, encoding="utf-8-sig")
print_basic(krx_raw, "krx_raw")

required_cols = ["stock_code", "corp_name", "market", "sector"]
missing_cols = [c for c in required_cols if c not in krx_raw.columns]
if missing_cols:
    raise ValueError(f"KRX_FILE missing required columns: {missing_cols}")

krx_all = krx_raw.copy()
krx_all["stock_code"] = krx_all["stock_code"].apply(clean_stock_code)
krx_all["corp_name"] = krx_all["corp_name"].astype(str).str.strip()
krx_all["market"] = krx_all["market"].astype(str).str.strip()
krx_all["sector"] = krx_all["sector"].astype(str).str.strip()
krx_all["is_standard_stock_code"] = krx_all["stock_code"].apply(is_standard_stock_code)

financial_keywords = [
    "금융", "은행", "보험", "증권", "투자", "신탁",
    "카드", "캐피탈", "리스", "부동산", "회사 본부", "기금",
    "기타 금융", "금융 지원", "보험 및 연금"
]

if "is_financial_conservative" in krx_all.columns:
    krx_all["is_financial_conservative"] = krx_all["is_financial_conservative"].astype(str).str.lower().map({
        "true": True, "false": False, "1": True, "0": False
    })
    krx_all["is_financial_conservative"] = krx_all["is_financial_conservative"].fillna(
        krx_all["sector"].apply(lambda x: any(k in x for k in financial_keywords))
    )
else:
    krx_all["is_financial_conservative"] = krx_all["sector"].apply(
        lambda x: any(k in x for k in financial_keywords)
    )

krx_kospi_all = krx_all[krx_all["market"] == "KOSPI"].copy()
krx_kospi_standard = krx_kospi_all[krx_kospi_all["is_standard_stock_code"]].copy()

krx_kospi_nonfin_before_dart = (
    krx_kospi_standard[~krx_kospi_standard["is_financial_conservative"]]
    .drop_duplicates("stock_code")
    .reset_index(drop=True)
    .copy()
)

print("KRX current KOSPI firms:", krx_kospi_all["stock_code"].nunique())
print("KOSPI standard code firms:", krx_kospi_standard["stock_code"].nunique())
print("KOSPI nonfinancial before DART:", krx_kospi_nonfin_before_dart["stock_code"].nunique())

save_df(krx_all, "01_krx_all_clean.csv")
save_df(krx_kospi_nonfin_before_dart, "02_krx_current_kospi_nonfinancial_before_dart.csv")

In [ ]:
# ============================================================
# Cell 5. Download corp codes and match with KRX current sample
# ============================================================

def download_corp_codes():
    zip_path = RAW_DIR / "corpCode.zip"
    xml_path = RAW_DIR / "CORPCODE.xml"
    
    if xml_path.exists():
        return xml_path
    
    url = f"{BASE_URL}/corpCode.xml"
    params = {"crtfc_key": DART_API_KEY}
    
    print("Downloading corpCode.xml...")
    r = requests.get(url, params=params, timeout=TIMEOUT)
    r.raise_for_status()
    
    with open(zip_path, "wb") as f:
        f.write(r.content)
    
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(RAW_DIR)
    
    if not xml_path.exists():
        candidates = list(RAW_DIR.glob("*.xml"))
        if candidates:
            xml_path = candidates[0]
        else:
            raise FileNotFoundError("CORPCODE.xml not found.")
    
    return xml_path


def parse_corp_codes(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    rows = []
    for item in root.findall("list"):
        row = {}
        for child in item:
            row[child.tag] = child.text
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df["stock_code"] = df["stock_code"].apply(clean_stock_code)
    df["corp_code"] = df["corp_code"].apply(clean_corp_code)
    return df


corp_xml = download_corp_codes()
corp_codes_all = parse_corp_codes(corp_xml)

corp_codes_listed = corp_codes_all[
    corp_codes_all["stock_code"].apply(is_standard_stock_code)
].drop_duplicates("stock_code").copy()

current_sample_with_dart = krx_kospi_nonfin_before_dart.merge(
    corp_codes_listed[["corp_code", "corp_name", "stock_code", "modify_date"]],
    on="stock_code",
    how="left",
    suffixes=("", "_dart")
)

current_sample_with_dart["dart_matched"] = current_sample_with_dart["corp_code"].notna()

current_sample_matched = (
    current_sample_with_dart[current_sample_with_dart["dart_matched"]]
    .drop_duplicates("corp_code")
    .reset_index(drop=True)
    .copy()
)

current_sample_matched["corp_code"] = current_sample_matched["corp_code"].apply(clean_corp_code)

if PILOT_MODE:
    current_sample_matched = current_sample_matched.head(PILOT_N_FIRMS).copy()

print("KOSPI nonfinancial before DART:", krx_kospi_nonfin_before_dart["stock_code"].nunique())
print("DART matched current sample:", current_sample_matched["corp_code"].nunique())
print("DART unmatched:", (~current_sample_with_dart["dart_matched"]).sum())

assert current_sample_matched["corp_code"].nunique() <= krx_kospi_nonfin_before_dart["stock_code"].nunique(), \
    "Matched firms cannot exceed pre-DART nonfinancial sample."

save_df(current_sample_with_dart, "03_current_sample_with_dart_matching_status.csv")
save_df(current_sample_matched, "04_current_sample_matched_final_firm_list.csv")

In [ ]:
# ============================================================
# Cell 6. Build annual DART business-report universe for representativeness
# ============================================================

def fetch_periodic_filings_by_period_no_corp_cls(bgn_de, end_de):
    """
    전체 정기공시를 기간별로 가져온 뒤 KRX stock_code로 KOSPI/비금융 필터링.
    corp_cls 조건을 넣지 않아 API 오류 가능성을 낮춤.
    """
    rows = []
    params = {
        "bgn_de": bgn_de,
        "end_de": end_de,
        "pblntf_ty": "A",
        "page_no": 1,
        "page_count": 100
    }
    
    while True:
        data = dart_get_json("list.json", params, use_cache=True)
        status = str(data.get("status", ""))
        
        if status == "000":
            rows.extend(data.get("list", []))
            total_page = int(data.get("total_page", 1) or 1)
            if int(params["page_no"]) >= total_page:
                break
            params["page_no"] += 1
        
        elif status == "013":
            break
        
        else:
            print(f"[WARN] {bgn_de}-{end_de}, status={status}, message={data.get('message')}")
            break
    
    return rows


universe_rows = []

for filing_year in tqdm(FILING_YEARS_FOR_UNIVERSE, desc="DART annual-report universe"):
    for bgn_de, end_de in month_ranges(filing_year):
        part = fetch_periodic_filings_by_period_no_corp_cls(bgn_de, end_de)
        for r in part:
            r["filing_year"] = filing_year
            r["query_bgn_de"] = bgn_de
            r["query_end_de"] = end_de
        universe_rows.extend(part)

dart_universe_filings = pd.DataFrame(universe_rows)

if dart_universe_filings.empty:
    raise ValueError("DART universe collection failed. Check API key or Open DART response.")

dart_universe_filings["corp_code"] = dart_universe_filings["corp_code"].apply(clean_corp_code)
dart_universe_filings["stock_code"] = dart_universe_filings.get(
    "stock_code", pd.Series(dtype=str)
).apply(clean_stock_code)
dart_universe_filings["report_nm"] = dart_universe_filings["report_nm"].astype(str)
dart_universe_filings["rcept_dt"] = dart_universe_filings["rcept_dt"].astype(str)

if "rcept_no" in dart_universe_filings.columns:
    dart_universe_filings = dart_universe_filings.drop_duplicates("rcept_no").copy()

# 사업보고서만
dart_universe_annual_filings = dart_universe_filings[
    dart_universe_filings["report_nm"].str.contains("사업보고서", na=False)
].copy()

dart_universe_annual_filings["is_correction"] = (
    dart_universe_annual_filings["report_nm"].str.contains("정정", na=False).astype(int)
)

# 제출연도 기준 추정 사업연도
# 일반적으로 사업보고서는 다음 해에 제출되므로 filing_year - 1
dart_universe_annual_filings["fiscal_year_est"] = dart_universe_annual_filings["filing_year"] - 1

# 분석기간 사업연도만
dart_universe_annual_filings = dart_universe_annual_filings[
    dart_universe_annual_filings["fiscal_year_est"].between(START_FISCAL_YEAR, END_FISCAL_YEAR)
].copy()

# KRX 정보 붙여서 KOSPI 비금융 필터링
dart_universe_annual_filings = dart_universe_annual_filings.merge(
    krx_all[["stock_code", "market", "sector", "is_financial_conservative"]],
    on="stock_code",
    how="left"
)

dart_universe_annual_kospi_nonfin = dart_universe_annual_filings[
    (dart_universe_annual_filings["market"] == "KOSPI") &
    (~dart_universe_annual_filings["is_financial_conservative"].fillna(False)) &
    (dart_universe_annual_filings["stock_code"].apply(is_standard_stock_code))
].copy()

# firm-year universe: 중복 공시행 제거 후 기업-사업연도 단위
annual_dart_universe_firmyear = (
    dart_universe_annual_kospi_nonfin
    .groupby(["fiscal_year_est", "corp_code"], as_index=False)
    .agg(
        corp_name=("corp_name", "first"),
        stock_code=("stock_code", "first"),
        market=("market", "first"),
        sector=("sector", "first"),
        n_annual_report_filings=("rcept_no", "nunique"),
        n_annual_corrections=("is_correction", "sum"),
        any_correction=("is_correction", "max")
    )
    .rename(columns={"fiscal_year_est": "fiscal_year"})
)

print_basic(annual_dart_universe_firmyear, "annual_dart_universe_firmyear")

annual_universe_summary = (
    annual_dart_universe_firmyear
    .groupby("fiscal_year", as_index=False)
    .agg(
        annual_universe_firms=("corp_code", "nunique"),
        annual_report_filings=("n_annual_report_filings", "sum"),
        annual_corrections=("n_annual_corrections", "sum")
    )
)

display(annual_universe_summary)

save_df(dart_universe_filings, "05_dart_periodic_filings_universe_raw.csv")
save_df(dart_universe_annual_filings, "06_dart_annual_filings_universe_all_markets.csv")
save_df(dart_universe_annual_kospi_nonfin, "07_dart_annual_filings_kospi_nonfin.csv")
save_df(annual_dart_universe_firmyear, "08_annual_dart_universe_firmyear_kospi_nonfin.csv")
save_df(annual_universe_summary, "09_annual_dart_universe_summary.csv")

In [ ]:
# ============================================================
# Cell 7. Correct coverage calculation
# ============================================================

current_sample_codes = set(current_sample_matched["corp_code"].astype(str))

annual_dart_universe_firmyear["in_current_sample"] = annual_dart_universe_firmyear["corp_code"].isin(current_sample_codes)

coverage_rows = []

for fy, g in annual_dart_universe_firmyear.groupby("fiscal_year"):
    annual_universe_firms = g["corp_code"].nunique()
    in_current_sample_firms = g.loc[g["in_current_sample"], "corp_code"].nunique()
    
    coverage_rows.append({
        "fiscal_year": int(fy),
        "annual_universe_firms": int(annual_universe_firms),
        "in_current_sample_firms": int(in_current_sample_firms),
        "coverage_rate_current_sample": in_current_sample_firms / annual_universe_firms if annual_universe_firms else np.nan
    })

coverage_by_year = pd.DataFrame(coverage_rows).sort_values("fiscal_year")

display(coverage_by_year)

if (coverage_by_year["coverage_rate_current_sample"] > 1.000001).any():
    raise ValueError("Coverage rate exceeds 100%. Duplicates or universe construction error remain.")

# current sample의 사업보고서 존재 여부
current_panel_check = (
    current_sample_matched[["corp_code", "stock_code", "corp_name", "market", "sector"]]
    .assign(_key=1)
    .merge(pd.DataFrame({"fiscal_year": FISCAL_YEARS, "_key": 1}), on="_key")
    .drop(columns="_key")
)

annual_universe_key = annual_dart_universe_firmyear[
    ["fiscal_year", "corp_code", "n_annual_report_filings"]
].drop_duplicates(["fiscal_year", "corp_code"])

current_panel_check = current_panel_check.merge(
    annual_universe_key,
    on=["fiscal_year", "corp_code"],
    how="left"
)

current_panel_check["has_annual_report_in_dart_universe"] = current_panel_check["n_annual_report_filings"].notna()

current_report_coverage = (
    current_panel_check
    .groupby("fiscal_year", as_index=False)
    .agg(
        current_sample_firms=("corp_code", "nunique"),
        firms_with_annual_report=("has_annual_report_in_dart_universe", "sum")
    )
)

current_report_coverage["annual_report_coverage_rate"] = (
    current_report_coverage["firms_with_annual_report"] / current_report_coverage["current_sample_firms"]
)

display(current_report_coverage)

save_df(coverage_by_year, "10_coverage_current_sample_vs_annual_dart_universe_CORRECTED.csv")
save_df(current_panel_check, "11_current_sample_annual_report_coverage_detail.csv")
save_df(current_report_coverage, "12_current_sample_annual_report_coverage_by_year.csv")

In [ ]:
# ============================================================
# Cell 8. Create analysis panel base with hard consistency checks
# ============================================================

if ANALYSIS_SAMPLE_MODE == "current_sample":
    analysis_firms = current_sample_matched.copy()
    analysis_firms = analysis_firms.drop_duplicates("corp_code").copy()
    
    analysis_panel_base = (
        analysis_firms[["corp_code", "stock_code", "corp_name", "market", "sector"]]
        .assign(_key=1)
        .merge(pd.DataFrame({"fiscal_year": FISCAL_YEARS, "_key": 1}), on="_key")
        .drop(columns="_key")
    )
    
    n_base_firms = analysis_panel_base["corp_code"].nunique()
    n_matched_firms = current_sample_matched["corp_code"].nunique()
    
    print("n_base_firms:", n_base_firms)
    print("n_matched_firms:", n_matched_firms)
    
    if n_base_firms != n_matched_firms:
        raise ValueError("analysis_panel_base firm count does not match current_sample_matched.")
    
    if n_base_firms > krx_kospi_nonfin_before_dart["stock_code"].nunique():
        raise ValueError("Final base firms cannot exceed pre-DART KOSPI nonfinancial firms.")

elif ANALYSIS_SAMPLE_MODE == "annual_dart_universe":
    analysis_panel_base = annual_dart_universe_firmyear[
        ["fiscal_year", "corp_code", "stock_code", "corp_name", "market", "sector"]
    ].drop_duplicates(["fiscal_year", "corp_code"]).copy()

else:
    raise ValueError("ANALYSIS_SAMPLE_MODE must be current_sample or annual_dart_universe.")

analysis_panel_base["corp_code"] = analysis_panel_base["corp_code"].apply(clean_corp_code)
analysis_panel_base["stock_code"] = analysis_panel_base["stock_code"].apply(clean_stock_code)

print_basic(analysis_panel_base, "analysis_panel_base")
print("analysis base firms:", analysis_panel_base["corp_code"].nunique())
print("analysis base firm-years:", len(analysis_panel_base))

save_df(analysis_panel_base, f"13_analysis_panel_base_{ANALYSIS_SAMPLE_MODE}.csv")

In [ ]:
# ============================================================
# Cell 9. Collect firm-year filings and build correction variables
# ============================================================

def fetch_firm_periodic_filings_for_fiscal_year(corp_code, fiscal_year):
    """
    fiscal_year 사업보고서는 보통 fiscal_year+1년에 제출.
    그래도 정정공시가 같은 해 말/다음 해 이후 있을 수 있어
    fiscal_year+1 전체 연도까지 검색.
    
    단순화:
    fiscal_year+1년 1월 1일 ~ 12월 31일 검색.
    """
    filing_year = int(fiscal_year) + 1
    rows = []
    
    params = {
        "corp_code": corp_code,
        "bgn_de": f"{filing_year}0101",
        "end_de": f"{filing_year}1231",
        "pblntf_ty": "A",
        "page_no": 1,
        "page_count": 100
    }
    
    while True:
        data = dart_get_json("list.json", params, use_cache=True)
        status = str(data.get("status", ""))
        
        if status == "000":
            rows.extend(data.get("list", []))
            total_page = int(data.get("total_page", 1) or 1)
            if int(params["page_no"]) >= total_page:
                break
            params["page_no"] += 1
        
        elif status == "013":
            break
        
        else:
            print(f"[WARN] corp={corp_code}, fiscal_year={fiscal_year}, status={status}, message={data.get('message')}")
            break
    
    for r in rows:
        r["fiscal_year"] = int(fiscal_year)
        r["filing_year"] = filing_year
    
    return rows


firm_years_to_fetch = analysis_panel_base[["corp_code", "fiscal_year"]].drop_duplicates()

filing_rows = []

for _, fy in tqdm(firm_years_to_fetch.iterrows(), total=len(firm_years_to_fetch), desc="firm-year filings"):
    filing_rows.extend(fetch_firm_periodic_filings_for_fiscal_year(fy["corp_code"], int(fy["fiscal_year"])))

analysis_filings = pd.DataFrame(filing_rows)

if analysis_filings.empty:
    raise ValueError("No filings collected for analysis sample.")

analysis_filings["corp_code"] = analysis_filings["corp_code"].apply(clean_corp_code)
analysis_filings["stock_code"] = analysis_filings.get("stock_code", pd.Series(dtype=str)).apply(clean_stock_code)
analysis_filings["report_nm"] = analysis_filings["report_nm"].astype(str)
analysis_filings["rcept_dt"] = analysis_filings["rcept_dt"].astype(str)

if "rcept_no" in analysis_filings.columns:
    analysis_filings = analysis_filings.drop_duplicates("rcept_no").copy()

analysis_annual_filings = analysis_filings[
    analysis_filings["report_nm"].str.contains("사업보고서", na=False)
].copy()

analysis_annual_filings["is_correction"] = analysis_annual_filings["report_nm"].str.contains("정정", na=False).astype(int)

analysis_annual_corrections = analysis_annual_filings[
    analysis_annual_filings["is_correction"] == 1
].copy()

correction_agg = (
    analysis_annual_corrections
    .groupby(["corp_code", "fiscal_year"], as_index=False)
    .agg(
        CorrectionCount=("rcept_no", "nunique"),
        FirstCorrectionDate=("rcept_dt", "min"),
        LastCorrectionDate=("rcept_dt", "max")
    )
)

panel_correction = analysis_panel_base.merge(
    correction_agg,
    on=["corp_code", "fiscal_year"],
    how="left"
)

panel_correction["CorrectionCount"] = panel_correction["CorrectionCount"].fillna(0).astype(int)
panel_correction["CorrectionDummy"] = (panel_correction["CorrectionCount"] > 0).astype(int)
panel_correction["LogCorrectionCount"] = np.log1p(panel_correction["CorrectionCount"])

print_basic(panel_correction, "panel_correction")
print(panel_correction[["CorrectionDummy", "CorrectionCount", "LogCorrectionCount"]].describe())

save_df(analysis_filings, f"14_all_filings_{ANALYSIS_SAMPLE_MODE}.csv")
save_df(analysis_annual_filings, f"15_annual_filings_{ANALYSIS_SAMPLE_MODE}.csv")
save_df(analysis_annual_corrections, f"16_annual_corrections_{ANALYSIS_SAMPLE_MODE}.csv")
save_df(panel_correction, f"17_panel_correction_{ANALYSIS_SAMPLE_MODE}.csv")

In [ ]:
# ============================================================
# Cell 10. Material correction classification and validation sample
# ============================================================

MATERIAL_KEYWORDS = [
    "재무제표", "연결재무제표", "재무상태표", "손익계산서", "포괄손익계산서",
    "현금흐름표", "자본변동표",
    "매출", "영업이익", "당기순이익", "자산총계", "부채총계", "자본총계",
    "감사의견", "감사인", "회계감사", "감사용역", "비감사용역",
    "내부회계관리제도",
    "최대주주", "소액주주", "사외이사", "이사회", "감사위원",
    "특수관계", "종속기업", "관계기업", "타법인", "출자"
]

def document_text_cache(rcept_no):
    return CACHE_DIR / f"document_text_{rcept_no}.txt"


def fetch_document_text(rcept_no):
    cp = document_text_cache(rcept_no)
    if cp.exists():
        return cp.read_text(encoding="utf-8", errors="ignore")
    
    url = f"{BASE_URL}/document.xml"
    params = {"crtfc_key": DART_API_KEY, "rcept_no": rcept_no}
    
    try:
        time.sleep(REQUEST_SLEEP)
        r = requests.get(url, params=params, timeout=TIMEOUT)
        r.raise_for_status()
        content = r.content
        
        text = ""
        if content[:2] == b"PK":
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                for name in z.namelist():
                    raw = z.read(name)
                    for enc in ["utf-8", "cp949", "euc-kr"]:
                        try:
                            text += raw.decode(enc, errors="ignore")
                            break
                        except:
                            pass
        else:
            for enc in ["utf-8", "cp949", "euc-kr"]:
                try:
                    text = content.decode(enc, errors="ignore")
                    break
                except:
                    pass
        
        cp.write_text(text, encoding="utf-8", errors="ignore")
        return text
    
    except Exception:
        return ""


material_rows = []

for _, r in tqdm(analysis_annual_corrections.iterrows(), total=len(analysis_annual_corrections), desc="material correction"):
    text = fetch_document_text(r["rcept_no"])
    hits = sorted(set([kw for kw in MATERIAL_KEYWORDS if kw in text]))
    
    material_rows.append({
        "corp_code": clean_corp_code(r["corp_code"]),
        "corp_name": r.get("corp_name"),
        "fiscal_year": int(r["fiscal_year"]),
        "filing_year": int(r["filing_year"]),
        "rcept_no": r["rcept_no"],
        "rcept_dt": r["rcept_dt"],
        "report_nm": r["report_nm"],
        "MaterialCorrection": int(len(hits) > 0),
        "MaterialKeywordHits": len(hits),
        "MaterialKeywords": "|".join(hits)
    })

material_detail = pd.DataFrame(material_rows)

if material_detail.empty:
    material_agg = pd.DataFrame(columns=["corp_code", "fiscal_year", "MaterialCorrectionCount", "MaterialKeywordHits"])
else:
    material_agg = (
        material_detail
        .groupby(["corp_code", "fiscal_year"], as_index=False)
        .agg(
            MaterialCorrectionCount=("MaterialCorrection", "sum"),
            MaterialKeywordHits=("MaterialKeywordHits", "sum")
        )
    )

panel_material = panel_correction.merge(
    material_agg,
    on=["corp_code", "fiscal_year"],
    how="left"
)

panel_material["MaterialCorrectionCount"] = panel_material["MaterialCorrectionCount"].fillna(0).astype(int)
panel_material["MaterialCorrectionDummy"] = (panel_material["MaterialCorrectionCount"] > 0).astype(int)
panel_material["MaterialKeywordHits"] = panel_material["MaterialKeywordHits"].fillna(0).astype(int)

# 수작업 검증 샘플
np.random.seed(4268)

if material_detail.empty:
    material_validation_sample = pd.DataFrame()
else:
    material_detail["period_group"] = np.where(material_detail["fiscal_year"] <= 2021, "2018-2021", "2022-2024")
    samples = []
    
    for period in ["2018-2021", "2022-2024"]:
        for mat in [0, 1]:
            tmp = material_detail[
                (material_detail["period_group"] == period) &
                (material_detail["MaterialCorrection"] == mat)
            ].copy()
            n = min(15, len(tmp))
            if n > 0:
                samples.append(tmp.sample(n=n, random_state=4268))
    
    material_validation_sample = pd.concat(samples, ignore_index=True) if samples else pd.DataFrame()

print_basic(panel_material, "panel_material")
print_basic(material_validation_sample, "material_validation_sample")

save_df(material_detail, f"18_material_corrections_detail_{ANALYSIS_SAMPLE_MODE}.csv")
save_df(panel_material, f"19_panel_material_{ANALYSIS_SAMPLE_MODE}.csv")
save_df(material_validation_sample, f"20_material_validation_sample_{ANALYSIS_SAMPLE_MODE}.csv")

In [ ]:
# ============================================================
# Cell 11. Collect periodic report key info
# ============================================================

ENDPOINTS = {
    "auditor_opinion": "accnutAdtorNmNdAdtOpinion.json",
    "audit_contract": "adtServcCnclsSttus.json",
    "non_audit_contract": "accnutAdtorNonAdtServcCnclsSttus.json",
    "outside_director": "outcmpnyDrctrNdChangeSttus.json",
    "largest_shareholder": "hyslrSttus.json",
    "minority_shareholder": "mrhlSttus.json",
}

def fetch_endpoint(endpoint, corp_code, fiscal_year):
    params = {
        "corp_code": corp_code,
        "bsns_year": str(int(fiscal_year)),
        "reprt_code": REPORT_CODE
    }
    return dart_list(endpoint, params, use_cache=True)


firm_years_info = panel_material[["corp_code", "fiscal_year"]].drop_duplicates()

endpoint_frames = {}

for label, endpoint in ENDPOINTS.items():
    rows = []
    print("\nCollecting:", label, endpoint)
    
    for _, fy in tqdm(firm_years_info.iterrows(), total=len(firm_years_info), desc=label):
        out = fetch_endpoint(endpoint, fy["corp_code"], int(fy["fiscal_year"]))
        for item in out:
            item["fiscal_year"] = int(fy["fiscal_year"])
            item["endpoint_label"] = label
            rows.append(item)
    
    df = pd.DataFrame(rows)
    if not df.empty and "corp_code" in df.columns:
        df["corp_code"] = df["corp_code"].apply(clean_corp_code)
    
    endpoint_frames[label] = df
    print_basic(df, f"raw_{label}", n=3)
    save_df(df, f"raw_{label}_{ANALYSIS_SAMPLE_MODE}.csv")

In [ ]:
# ============================================================
# Cell 12. Process auditor and audit fee variables
# ============================================================

aud = endpoint_frames.get("auditor_opinion", pd.DataFrame()).copy()

if not aud.empty:
    aud["fiscal_year"] = aud["fiscal_year"].astype(int)
    aud["corp_code"] = aud["corp_code"].apply(clean_corp_code)
    
    auditor_col = first_existing_col(aud, ["auditor_nm", "adtor", "auditor", "nm", "adt_instt_nm"])
    opinion_col = first_existing_col(aud, ["adt_opinion", "audit_opinion", "opinion"])
    
    auditor_vars = aud[["corp_code", "fiscal_year"]].drop_duplicates().copy()
    
    if auditor_col:
        tmp = aud[["corp_code", "fiscal_year", auditor_col]].dropna().drop_duplicates(["corp_code", "fiscal_year"])
        tmp = tmp.rename(columns={auditor_col: "AuditorName"})
        auditor_vars = auditor_vars.merge(tmp, on=["corp_code", "fiscal_year"], how="left")
    else:
        auditor_vars["AuditorName"] = np.nan
    
    if opinion_col:
        tmp = aud[["corp_code", "fiscal_year", opinion_col]].dropna().drop_duplicates(["corp_code", "fiscal_year"])
        tmp = tmp.rename(columns={opinion_col: "AuditOpinion"})
        auditor_vars = auditor_vars.merge(tmp, on=["corp_code", "fiscal_year"], how="left")
    else:
        auditor_vars["AuditOpinion"] = np.nan
    
    big4_keywords = ["삼일", "삼정", "안진", "한영", "pwc", "kpmg", "deloitte", "ey"]
    auditor_vars["Big4Auditor"] = auditor_vars["AuditorName"].astype(str).apply(
        lambda x: int(any(k in x.lower() for k in big4_keywords))
    )
    auditor_vars["CleanOpinion"] = auditor_vars["AuditOpinion"].astype(str).str.contains("적정", na=False).astype(int)
else:
    auditor_vars = pd.DataFrame(columns=["corp_code", "fiscal_year", "AuditorName", "AuditOpinion", "Big4Auditor", "CleanOpinion"])


audit = endpoint_frames.get("audit_contract", pd.DataFrame()).copy()

if not audit.empty:
    audit["fiscal_year"] = audit["fiscal_year"].astype(int)
    audit["corp_code"] = audit["corp_code"].apply(clean_corp_code)
    
    fee_col = first_existing_col(audit, ["adt_servc_cncls_mendng", "servc_mendng", "mendng", "cntrct_mendng"])
    hour_col = first_existing_col(audit, ["adt_servc_cncls_time", "servc_time", "tot_reqre_time"])
    
    audit["AuditFee"] = audit[fee_col].apply(parse_number) if fee_col else np.nan
    audit["AuditHours"] = audit[hour_col].apply(parse_number) if hour_col else np.nan
    
    audit.loc[audit["AuditFee"] < 0, "AuditFee"] = np.nan
    audit.loc[audit["AuditFee"] > 1e12, "AuditFee"] = np.nan
    
    audit_fee_vars = (
        audit
        .groupby(["corp_code", "fiscal_year"], as_index=False)
        .agg(AuditFee=("AuditFee", "sum"), AuditHours=("AuditHours", "sum"))
    )
    
    audit_fee_vars.loc[audit_fee_vars["AuditFee"] <= 0, "AuditFee"] = np.nan
    audit_fee_vars.loc[audit_fee_vars["AuditHours"] <= 0, "AuditHours"] = np.nan
else:
    audit_fee_vars = pd.DataFrame(columns=["corp_code", "fiscal_year", "AuditFee", "AuditHours"])


non = endpoint_frames.get("non_audit_contract", pd.DataFrame()).copy()

if not non.empty:
    non["fiscal_year"] = non["fiscal_year"].astype(int)
    non["corp_code"] = non["corp_code"].apply(clean_corp_code)
    
    non_fee_col = first_existing_col(non, ["servc_mendng", "mendng", "cntrct_mendng", "service_fee"])
    
    non["NonAuditFee"] = non[non_fee_col].apply(parse_number) if non_fee_col else np.nan
    non.loc[non["NonAuditFee"] < 0, "NonAuditFee"] = np.nan
    non.loc[non["NonAuditFee"] > 1e12, "NonAuditFee"] = np.nan
    
    non_audit_vars = (
        non
        .groupby(["corp_code", "fiscal_year"], as_index=False)
        .agg(
            NonAuditContractCount=("corp_code", "size"),
            NonAuditFee=("NonAuditFee", "sum")
        )
    )
    
    non_audit_vars.loc[non_audit_vars["NonAuditFee"] <= 0, "NonAuditFee"] = 0
else:
    non_audit_vars = pd.DataFrame(columns=["corp_code", "fiscal_year", "NonAuditContractCount", "NonAuditFee"])

print_basic(auditor_vars, "auditor_vars")
print_basic(audit_fee_vars, "audit_fee_vars")
print_basic(non_audit_vars, "non_audit_vars")

In [ ]:
# ============================================================
# Cell 13. Process governance and ownership variables
# ============================================================

outside = endpoint_frames.get("outside_director", pd.DataFrame()).copy()

if not outside.empty:
    outside["fiscal_year"] = outside["fiscal_year"].astype(int)
    outside["corp_code"] = outside["corp_code"].apply(clean_corp_code)
    
    board_col = first_existing_col(outside, ["drctr_co", "director_co", "tot_drctr_co"])
    outside_col = first_existing_col(outside, ["otcmp_drctr_co", "outside_director_co", "outcmpny_drctr_co"])
    
    outside["BoardSize_raw"] = outside[board_col].apply(parse_number) if board_col else np.nan
    outside["OutsideDirectorCount_raw"] = outside[outside_col].apply(parse_number) if outside_col else np.nan
    
    outside_tmp = outside[["corp_code", "fiscal_year", "BoardSize_raw", "OutsideDirectorCount_raw"]].dropna(
        subset=["BoardSize_raw", "OutsideDirectorCount_raw"],
        how="all"
    )
    
    outside_vars = (
        outside_tmp
        .groupby(["corp_code", "fiscal_year"], as_index=False)
        .agg(
            BoardSize=("BoardSize_raw", "max"),
            OutsideDirectorCount=("OutsideDirectorCount_raw", "max")
        )
    )
    
    outside_vars["OutsideDirectorRatio"] = outside_vars.apply(
        lambda r: safe_divide(r["OutsideDirectorCount"], r["BoardSize"]),
        axis=1
    )
    
    outside_vars.loc[outside_vars["BoardSize"] <= 0, "BoardSize"] = np.nan
    outside_vars.loc[outside_vars["OutsideDirectorRatio"] < 0, "OutsideDirectorRatio"] = np.nan
    outside_vars.loc[outside_vars["OutsideDirectorRatio"] > 1, "OutsideDirectorRatio"] = np.nan
else:
    outside_vars = pd.DataFrame(columns=["corp_code", "fiscal_year", "BoardSize", "OutsideDirectorCount", "OutsideDirectorRatio"])


largest = endpoint_frames.get("largest_shareholder", pd.DataFrame()).copy()

if not largest.empty:
    largest["fiscal_year"] = largest["fiscal_year"].astype(int)
    largest["corp_code"] = largest["corp_code"].apply(clean_corp_code)
    
    own_col = first_existing_col(largest, [
        "trmend_posesn_stock_qota_rt", "posesn_stock_qota_rt",
        "qota_rt", "stock_qota_rt", "rt"
    ])
    
    largest["LargestShareholderOwnership"] = largest[own_col].apply(parse_number) if own_col else np.nan
    
    largest_vars = (
        largest
        .groupby(["corp_code", "fiscal_year"], as_index=False)
        .agg(LargestShareholderOwnership=("LargestShareholderOwnership", "max"))
    )
    
    largest_vars.loc[largest_vars["LargestShareholderOwnership"] < 0, "LargestShareholderOwnership"] = np.nan
    largest_vars.loc[largest_vars["LargestShareholderOwnership"] > 100, "LargestShareholderOwnership"] = np.nan
else:
    largest_vars = pd.DataFrame(columns=["corp_code", "fiscal_year", "LargestShareholderOwnership"])


minority = endpoint_frames.get("minority_shareholder", pd.DataFrame()).copy()

if not minority.empty:
    minority["fiscal_year"] = minority["fiscal_year"].astype(int)
    minority["corp_code"] = minority["corp_code"].apply(clean_corp_code)
    
    ratio_col = first_existing_col(minority, ["mrhl_rate", "qota_rt", "rt"])
    count_col = first_existing_col(minority, ["shrholdr_co", "mrhl_co", "stockholdr_co"])
    
    minority["MinorityShareholderRatio"] = minority[ratio_col].apply(parse_number) if ratio_col else np.nan
    minority["MinorityShareholderCount"] = minority[count_col].apply(parse_number) if count_col else np.nan
    
    minority_vars = (
        minority
        .groupby(["corp_code", "fiscal_year"], as_index=False)
        .agg(
            MinorityShareholderRatio=("MinorityShareholderRatio", "max"),
            MinorityShareholderCount=("MinorityShareholderCount", "max")
        )
    )
    
    minority_vars.loc[minority_vars["MinorityShareholderRatio"] < 0, "MinorityShareholderRatio"] = np.nan
    minority_vars.loc[minority_vars["MinorityShareholderRatio"] > 100, "MinorityShareholderRatio"] = np.nan
else:
    minority_vars = pd.DataFrame(columns=["corp_code", "fiscal_year", "MinorityShareholderRatio", "MinorityShareholderCount"])

print_basic(outside_vars, "outside_vars")
print_basic(largest_vars, "largest_vars")
print_basic(minority_vars, "minority_vars")

In [ ]:
# ============================================================
# Cell 14. Financial statements and controls
# ============================================================

def fetch_financial_accounts(corp_code, fiscal_year):
    params = {
        "corp_code": corp_code,
        "bsns_year": str(int(fiscal_year)),
        "reprt_code": REPORT_CODE
    }
    return dart_list("fnlttSinglAcnt.json", params, use_cache=True)


fin_rows = []

for _, fy in tqdm(firm_years_info.iterrows(), total=len(firm_years_info), desc="financials"):
    out = fetch_financial_accounts(fy["corp_code"], int(fy["fiscal_year"]))
    for item in out:
        item["fiscal_year"] = int(fy["fiscal_year"])
        fin_rows.append(item)

fin_raw = pd.DataFrame(fin_rows)

if not fin_raw.empty:
    fin_raw["corp_code"] = fin_raw["corp_code"].apply(clean_corp_code)

save_df(fin_raw, f"raw_financial_accounts_{ANALYSIS_SAMPLE_MODE}.csv")
print_basic(fin_raw, "fin_raw")


def select_account_amount(df, patterns, prefer_sj=None):
    if df.empty:
        return pd.DataFrame(columns=["corp_code", "fiscal_year", "value"])
    
    d = df.copy()
    d["account_nm"] = d["account_nm"].astype(str)
    
    mask = False
    for p in patterns:
        mask = mask | d["account_nm"].str.contains(p, na=False, regex=False)
    
    d = d[mask].copy()
    
    if prefer_sj and "sj_div" in d.columns:
        d = d[d["sj_div"].eq(prefer_sj)].copy()
    
    if d.empty:
        return pd.DataFrame(columns=["corp_code", "fiscal_year", "value"])
    
    d["amount"] = d["thstrm_amount"].apply(parse_number)
    
    if "fs_div" in d.columns:
        d["fs_priority"] = np.where(d["fs_div"].eq("CFS"), 2, 1)
    else:
        d["fs_priority"] = 1
    
    def account_priority(name):
        for i, p in enumerate(patterns):
            if p in str(name):
                return i
        return 999
    
    d["account_priority"] = d["account_nm"].apply(account_priority)
    d["nonmissing_amount"] = d["amount"].notna().astype(int)
    
    d = d.sort_values(
        ["corp_code", "fiscal_year", "nonmissing_amount", "fs_priority", "account_priority"],
        ascending=[True, True, False, False, True]
    )
    
    out = d.groupby(["corp_code", "fiscal_year"], as_index=False)["amount"].first()
    out = out.rename(columns={"amount": "value"})
    return out


def make_fin_vars(fin_raw):
    if fin_raw.empty:
        return pd.DataFrame(columns=["corp_code", "fiscal_year"])
    
    df = fin_raw.copy()
    df["fiscal_year"] = df["fiscal_year"].astype(int)
    
    base = df[["corp_code", "fiscal_year"]].drop_duplicates()
    
    specs = {
        "Assets": {"patterns": ["자산총계"], "sj": "BS"},
        "Liabilities": {"patterns": ["부채총계"], "sj": "BS"},
        "Equity": {"patterns": ["자본총계"], "sj": "BS"},
        "Revenue": {"patterns": ["매출액", "영업수익", "수익(매출액)", "매출"], "sj": "IS"},
        "OperatingIncome": {"patterns": ["영업이익"], "sj": "IS"},
        "NetIncome": {
            "patterns": [
                "당기순이익", "당기순이익(손실)", "연결당기순이익",
                "지배기업의 소유주에게 귀속되는 당기순이익",
                "지배기업소유주지분순이익", "순이익"
            ],
            "sj": "IS"
        }
    }
    
    out = base.copy()
    
    for var, spec in specs.items():
        tmp = select_account_amount(df, spec["patterns"], prefer_sj=spec["sj"])
        tmp = tmp.rename(columns={"value": var})
        out = out.merge(tmp, on=["corp_code", "fiscal_year"], how="left")
    
    out["LogAssets"] = np.log(out["Assets"].replace(0, np.nan))
    out["Leverage"] = out.apply(lambda r: safe_divide(r["Liabilities"], r["Assets"]), axis=1)
    out["ROA"] = out.apply(lambda r: safe_divide(r["NetIncome"], r["Assets"]), axis=1)
    out["Loss"] = np.where(out["NetIncome"].notna(), (out["NetIncome"] < 0).astype(int), np.nan)
    
    out.loc[out["Assets"] <= 0, ["Assets", "LogAssets"]] = np.nan
    out.loc[(out["Leverage"] < 0) | (out["Leverage"] > 2), "Leverage"] = np.nan
    out.loc[(out["ROA"] < -2) | (out["ROA"] > 2), "ROA"] = np.nan
    
    return out


fin_vars = make_fin_vars(fin_raw)
print_basic(fin_vars, "fin_vars")
save_df(fin_vars, f"financial_vars_{ANALYSIS_SAMPLE_MODE}.csv")

In [ ]:
# ============================================================
# Cell 15. Merge final panel and hard sample checks
# ============================================================

final_panel = panel_material.copy()

for df in [
    auditor_vars,
    audit_fee_vars,
    non_audit_vars,
    outside_vars,
    largest_vars,
    minority_vars,
    fin_vars
]:
    if df is not None and not df.empty:
        final_panel = final_panel.merge(df, on=["corp_code", "fiscal_year"], how="left")

# 파생변수
final_panel["NonAuditFee"] = final_panel["NonAuditFee"].fillna(0) if "NonAuditFee" in final_panel.columns else 0
final_panel["NonAuditContractCount"] = (
    final_panel["NonAuditContractCount"].fillna(0) if "NonAuditContractCount" in final_panel.columns else 0
)

final_panel["NonAuditFeeRatio"] = final_panel.apply(
    lambda r: safe_divide(r.get("NonAuditFee", np.nan), r.get("AuditFee", np.nan)),
    axis=1
)

final_panel.loc[final_panel["NonAuditFeeRatio"] < 0, "NonAuditFeeRatio"] = np.nan
final_panel.loc[final_panel["NonAuditFeeRatio"] > 5, "NonAuditFeeRatio"] = np.nan

final_panel["LogAuditFee"] = np.log(final_panel["AuditFee"].replace(0, np.nan)) if "AuditFee" in final_panel.columns else np.nan
final_panel["LogNonAuditFee"] = np.log1p(final_panel["NonAuditFee"])
final_panel["LogMinorityShareholderCount"] = np.log1p(final_panel["MinorityShareholderCount"]) if "MinorityShareholderCount" in final_panel.columns else np.nan

for c in [
    "NonAuditFeeRatio", "LogAuditFee", "LogNonAuditFee",
    "OutsideDirectorRatio", "LargestShareholderOwnership",
    "MinorityShareholderRatio", "MinorityShareholderCount",
    "LogMinorityShareholderCount",
    "LogAssets", "Leverage", "ROA", "LogCorrectionCount"
]:
    if c in final_panel.columns:
        final_panel[c + "_w"] = winsorize_series(final_panel[c])

# ------------------------------------------------------------
# Hard checks: 692 → 711 같은 문제 방지
# ------------------------------------------------------------

n_final_firms = final_panel["corp_code"].nunique()
n_final_firmyears = len(final_panel)
n_matched_firms = current_sample_matched["corp_code"].nunique()
n_nonfin_before_dart = krx_kospi_nonfin_before_dart["stock_code"].nunique()

if ANALYSIS_SAMPLE_MODE == "current_sample":
    assert n_final_firms == n_matched_firms, (
        f"ERROR: final firms ({n_final_firms}) != DART matched current sample ({n_matched_firms}). "
        "Restart kernel and rerun all cells."
    )
    
    assert n_final_firms <= n_nonfin_before_dart, (
        f"ERROR: final firms ({n_final_firms}) > KOSPI nonfinancial before DART ({n_nonfin_before_dart})."
    )
    
    expected_firmyears = n_final_firms * len(FISCAL_YEARS)
    assert n_final_firmyears == expected_firmyears, (
        f"ERROR: final firm-years ({n_final_firmyears}) != firms*years ({expected_firmyears})."
    )

print_basic(final_panel, "final_panel")
print("n_final_firms:", n_final_firms)
print("n_final_firmyears:", n_final_firmyears)

save_df(final_panel, f"21_final_panel_{ANALYSIS_SAMPLE_MODE}_CLEAN.csv")

In [ ]:
# ============================================================
# Cell 16. Sample selection and representativeness tables
# ============================================================

sample_selection_final = pd.DataFrame([
    {
        "step": "KRX current KOSPI firms",
        "n": krx_kospi_all["stock_code"].nunique()
    },
    {
        "step": "KOSPI firms with standard 6-digit stock code",
        "n": krx_kospi_standard["stock_code"].nunique()
    },
    {
        "step": "KOSPI non-financial firms before DART matching",
        "n": krx_kospi_nonfin_before_dart["stock_code"].nunique()
    },
    {
        "step": "KOSPI non-financial firms with DART corp_code",
        "n": current_sample_matched["corp_code"].nunique()
    },
    {
        "step": f"Final analysis firms ({ANALYSIS_SAMPLE_MODE})",
        "n": final_panel["corp_code"].nunique()
    },
    {
        "step": f"Final analysis firm-years ({ANALYSIS_SAMPLE_MODE})",
        "n": len(final_panel)
    }
])

display(sample_selection_final)

if ANALYSIS_SAMPLE_MODE == "current_sample":
    n_dart_matched = int(sample_selection_final.loc[
        sample_selection_final["step"] == "KOSPI non-financial firms with DART corp_code", "n"
    ].iloc[0])
    n_final = int(sample_selection_final.loc[
        sample_selection_final["step"].str.startswith("Final analysis firms"), "n"
    ].iloc[0])
    
    if n_final > n_dart_matched:
        raise ValueError("Final sample cannot exceed DART matched sample.")

# 연도별 정정공시
yearly_correction_summary = (
    final_panel
    .groupby("fiscal_year", as_index=False)
    .agg(
        n_firm_years=("corp_code", "count"),
        n_firms=("corp_code", "nunique"),
        correction_rate=("CorrectionDummy", "mean"),
        avg_correction_count=("CorrectionCount", "mean"),
        avg_log_correction_count=("LogCorrectionCount", "mean"),
        material_rate=("MaterialCorrectionDummy", "mean")
    )
)

display(yearly_correction_summary)

# 기초통계
desc_vars = [
    "CorrectionDummy", "CorrectionCount", "LogCorrectionCount",
    "MaterialCorrectionDummy", "MaterialCorrectionCount",
    "OutsideDirectorRatio", "LargestShareholderOwnership",
    "MinorityShareholderCount", "LogMinorityShareholderCount",
    "LogAssets", "Leverage", "ROA", "Loss",
    "Big4Auditor", "CleanOpinion",
    "AuditFee", "NonAuditFee", "NonAuditFeeRatio"
]
desc_vars = [c for c in desc_vars if c in final_panel.columns]

descriptive_statistics = final_panel[desc_vars].describe().T
missing_rates = final_panel[desc_vars].isna().mean().sort_values(ascending=False).to_frame("missing_rate")

display(descriptive_statistics)
display(missing_rates)

sector_distribution = (
    final_panel
    .groupby("sector", as_index=False)
    .agg(
        n_firm_years=("corp_code", "count"),
        n_firms=("corp_code", "nunique"),
        correction_rate=("CorrectionDummy", "mean"),
        material_rate=("MaterialCorrectionDummy", "mean")
    )
    .sort_values("n_firms", ascending=False)
)

save_df(sample_selection_final, "22_sample_selection_final_CLEAN.csv")
save_df(yearly_correction_summary, "23_yearly_correction_summary_CLEAN.csv")
save_df(descriptive_statistics.reset_index().rename(columns={"index": "variable"}), "24_descriptive_statistics_CLEAN.csv")
save_df(missing_rates.reset_index().rename(columns={"index": "variable"}), "25_missing_rates_CLEAN.csv")
save_df(sector_distribution, "26_sector_distribution_CLEAN.csv")

In [ ]:
# ============================================================
# Cell 17. Regression preparation and functions
# ============================================================

reg = final_panel.copy()
reg = reg.replace([np.inf, -np.inf], np.nan)

base_controls = [c for c in ["LogAssets_w", "Leverage_w", "ROA_w", "Loss"] if c in reg.columns]

governance_vars = []
if "OutsideDirectorRatio_w" in reg.columns and reg["OutsideDirectorRatio_w"].notna().sum() > 100:
    governance_vars.append("OutsideDirectorRatio_w")
if "LargestShareholderOwnership_w" in reg.columns and reg["LargestShareholderOwnership_w"].notna().sum() > 100:
    governance_vars.append("LargestShareholderOwnership_w")
if "LogMinorityShareholderCount_w" in reg.columns and reg["LogMinorityShareholderCount_w"].notna().sum() > 100:
    governance_vars.append("LogMinorityShareholderCount_w")

auditor_basic_vars = []
if "Big4Auditor" in reg.columns and reg["Big4Auditor"].notna().sum() > 100:
    auditor_basic_vars.append("Big4Auditor")
if "CleanOpinion" in reg.columns and reg["CleanOpinion"].notna().sum() > 100:
    auditor_basic_vars.append("CleanOpinion")

audit_fee_vars = []
if "NonAuditFeeRatio_w" in reg.columns and reg["NonAuditFeeRatio_w"].notna().sum() > 300:
    audit_fee_vars.append("NonAuditFeeRatio_w")
if "LogAuditFee_w" in reg.columns and reg["LogAuditFee_w"].notna().sum() > 300:
    audit_fee_vars.append("LogAuditFee_w")
if "LogNonAuditFee_w" in reg.columns and reg["LogNonAuditFee_w"].notna().sum() > 300:
    audit_fee_vars.append("LogNonAuditFee_w")

print("base_controls:", base_controls)
print("governance_vars:", governance_vars)
print("auditor_basic_vars:", auditor_basic_vars)
print("audit_fee_vars:", audit_fee_vars)

regression_results = {}
regression_samples = {}

def run_ols_model(dep, xvars, name, binary_dep=False):
    use_cols = [dep, "corp_code", "fiscal_year", "sector"] + xvars
    use_cols = [c for c in use_cols if c in reg.columns]
    
    df = reg[use_cols].dropna().copy()
    
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("N:", len(df))
    print("Firms:", df["corp_code"].nunique() if len(df) else np.nan)
    print("Dep mean:", df[dep].mean() if len(df) else np.nan)
    
    if len(df) < 50:
        print("[SKIPPED] too few observations")
        return None, df
    
    if binary_dep and df[dep].nunique() < 2:
        print("[SKIPPED] no dependent variable variation")
        return None, df
    
    rhs = " + ".join(xvars + ["C(fiscal_year)", "C(sector)"])
    formula = f"{dep} ~ {rhs}"
    
    model = smf.ols(formula, data=df).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["corp_code"]}
    )
    
    print(model.summary())
    return model, df

In [ ]:
# ============================================================
# Cell 18. Main regressions
# ============================================================

models_to_run = [
    ("M1_controls", "CorrectionDummy", base_controls, True),
    ("M2_governance", "CorrectionDummy", base_controls + governance_vars, True),
    ("M3_governance_auditor_basic", "CorrectionDummy", base_controls + governance_vars + auditor_basic_vars, True),
    ("M4_audit_fee_aux", "CorrectionDummy", base_controls + governance_vars + auditor_basic_vars + audit_fee_vars, True),
    
    ("M5_material_controls", "MaterialCorrectionDummy", base_controls, True),
    ("M6_material_governance", "MaterialCorrectionDummy", base_controls + governance_vars, True),
    ("M7_material_auditor_basic", "MaterialCorrectionDummy", base_controls + governance_vars + auditor_basic_vars, True),
    
    ("M8_log_count_controls", "LogCorrectionCount", base_controls, False),
    ("M9_log_count_governance", "LogCorrectionCount", base_controls + governance_vars, False),
    ("M10_log_count_auditor_basic", "LogCorrectionCount", base_controls + governance_vars + auditor_basic_vars, False),
]

for name, dep, xvars, binary_dep in models_to_run:
    model, df = run_ols_model(dep, xvars, name, binary_dep=binary_dep)
    regression_results[name] = model
    regression_samples[name] = df

In [ ]:
# ============================================================
# Cell 19. Lagged governance robustness
# ============================================================

lag_panel = final_panel.sort_values(["corp_code", "fiscal_year"]).copy()

lag_original_vars = [
    "OutsideDirectorRatio_w",
    "LargestShareholderOwnership_w",
    "LogMinorityShareholderCount_w",
    "LogAssets_w",
    "Leverage_w",
    "ROA_w",
    "Loss"
]

for v in lag_original_vars:
    if v in lag_panel.columns:
        lag_panel[v + "_lag1"] = lag_panel.groupby("corp_code")[v].shift(1)

reg_lag = lag_panel.replace([np.inf, -np.inf], np.nan).copy()

lag_base_controls = [
    v for v in ["LogAssets_w_lag1", "Leverage_w_lag1", "ROA_w_lag1", "Loss_lag1"]
    if v in reg_lag.columns
]

lag_governance_vars = [
    v for v in [
        "OutsideDirectorRatio_w_lag1",
        "LargestShareholderOwnership_w_lag1",
        "LogMinorityShareholderCount_w_lag1"
    ]
    if v in reg_lag.columns and reg_lag[v].notna().sum() > 100
]

print("lag_base_controls:", lag_base_controls)
print("lag_governance_vars:", lag_governance_vars)

def run_lag_model(dep, xvars, name):
    use_cols = [dep, "corp_code", "fiscal_year", "sector"] + xvars
    use_cols = [c for c in use_cols if c in reg_lag.columns]
    
    df = reg_lag[use_cols].dropna().copy()
    
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("N:", len(df))
    print("Firms:", df["corp_code"].nunique() if len(df) else np.nan)
    print("Dep mean:", df[dep].mean() if len(df) else np.nan)
    
    if len(df) < 50 or df[dep].nunique() < 2:
        print("[SKIPPED]")
        return None, df
    
    rhs = " + ".join(xvars + ["C(fiscal_year)", "C(sector)"])
    formula = f"{dep} ~ {rhs}"
    
    model = smf.ols(formula, data=df).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["corp_code"]}
    )
    
    print(model.summary())
    return model, df


lag1, df_lag1 = run_lag_model(
    dep="CorrectionDummy",
    xvars=lag_base_controls + lag_governance_vars,
    name="LAG1_correction_lagged_governance"
)
regression_results["LAG1_correction_lagged_governance"] = lag1
regression_samples["LAG1_correction_lagged_governance"] = df_lag1

lag2, df_lag2 = run_lag_model(
    dep="MaterialCorrectionDummy",
    xvars=lag_base_controls + lag_governance_vars,
    name="LAG2_material_lagged_governance"
)
regression_results["LAG2_material_lagged_governance"] = lag2
regression_samples["LAG2_material_lagged_governance"] = df_lag2

In [ ]:
# ============================================================
# Cell 20. Save regression tables
# ============================================================

coef_tables = []

for name, model in regression_results.items():
    if model is None:
        continue
    
    tmp = pd.DataFrame({
        "model": name,
        "variable": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "t_or_z": model.tvalues.values,
        "p_value": model.pvalues.values,
        "nobs": model.nobs
    })
    coef_tables.append(tmp)

if coef_tables:
    coef_table = pd.concat(coef_tables, ignore_index=True)
else:
    coef_table = pd.DataFrame(columns=["model", "variable", "coef", "std_err", "t_or_z", "p_value", "nobs"])

coef_table["stars"] = coef_table["p_value"].apply(stars)
coef_table["coef_stars"] = coef_table.apply(lambda r: f"{r['coef']:.4f}{r['stars']}", axis=1)
coef_table["std_err_fmt"] = coef_table["std_err"].apply(lambda x: f"({x:.4f})" if pd.notna(x) else "")

sample_rows = []

for name, df in regression_samples.items():
    dep_col = df.columns[0] if len(df.columns) > 0 and not df.empty else None
    sample_rows.append({
        "model": name,
        "n_obs": len(df),
        "n_firms": df["corp_code"].nunique() if "corp_code" in df.columns and not df.empty else np.nan,
        "dep_mean": df[dep_col].mean() if dep_col and not df.empty else np.nan
    })

regression_sample_sizes = pd.DataFrame(sample_rows)

important_vars = [
    "OutsideDirectorRatio_w",
    "LargestShareholderOwnership_w",
    "LogMinorityShareholderCount_w",
    "LogAssets_w",
    "Leverage_w",
    "ROA_w",
    "Loss",
    "Big4Auditor",
    "CleanOpinion",
    "NonAuditFeeRatio_w",
    "LogAuditFee_w",
    "LogNonAuditFee_w",
    "OutsideDirectorRatio_w_lag1",
    "LargestShareholderOwnership_w_lag1",
    "LogMinorityShareholderCount_w_lag1",
    "LogAssets_w_lag1",
    "Leverage_w_lag1",
    "ROA_w_lag1",
    "Loss_lag1"
]

paper_coef = coef_table[coef_table["variable"].isin(important_vars)].copy()

display(coef_table)
display(regression_sample_sizes)
display(paper_coef)

save_df(coef_table, "27_regression_coefficients_all_CLEAN.csv")
save_df(regression_sample_sizes, "28_regression_sample_sizes_CLEAN.csv")
save_df(paper_coef, "29_paper_style_key_coefficients_long_CLEAN.csv")

pivot_coef = paper_coef.pivot_table(
    index="variable",
    columns="model",
    values="coef_stars",
    aggfunc="first"
)
display(pivot_coef)
pivot_coef.to_csv(OUT_DIR / "30_paper_style_key_coefficients_wide_CLEAN.csv", encoding="utf-8-sig")

for name, model in regression_results.items():
    if model is not None:
        with open(OUT_DIR / f"regression_{name}_CLEAN.txt", "w", encoding="utf-8") as f:
            f.write(model.summary().as_text())

In [ ]:
# ============================================================
# Cell 21. Literature review matrix skeleton
# ============================================================

literature_matrix = pd.DataFrame([
    {
        "category": "Foreign - Restatement",
        "paper": "Palmrose, Richardson, and Scholz (2004)",
        "journal": "Journal of Accounting and Economics",
        "topic": "Market reactions to restatement announcements",
        "connection_to_our_study": "Restatement/correction disclosures can be economically meaningful reporting-quality signals.",
        "our_difference": "Korean Open DART business-report correction disclosures, broader than US financial restatements."
    },
    {
        "category": "Foreign - Governance",
        "paper": "Abbott, Parker, and Peters (2004)",
        "journal": "Auditing: A Journal of Practice & Theory",
        "topic": "Audit committee characteristics and restatements",
        "connection_to_our_study": "Links governance mechanisms to financial reporting failures.",
        "our_difference": "Use Korean ownership and outside-director variables with DART corrections."
    },
    {
        "category": "Foreign - Directors",
        "paper": "Srinivasan (2005)",
        "journal": "Journal of Accounting Research",
        "topic": "Consequences of reporting failure for outside directors",
        "connection_to_our_study": "Correction disclosures can be governance-relevant reporting events.",
        "our_difference": "We examine whether governance predicts correction disclosures."
    },
    {
        "category": "Domestic - Correction Disclosure",
        "paper": "손성규, 재무제표 정정보고의 현황과 의미",
        "journal": "국내 회계학 문헌",
        "topic": "Financial statement correction reports",
        "connection_to_our_study": "Domestic precedent for correction disclosure research.",
        "our_difference": "Open DART automated panel and ownership/governance focus."
    },
    {
        "category": "Domestic - Analyst Information",
        "paper": "윤종철 (2024), 사업보고서 정정공시와 재무분석가 예측",
        "journal": "국내 학술지",
        "topic": "Annual report corrections and analyst forecasts",
        "connection_to_our_study": "Supports future information-environment analysis.",
        "our_difference": "We focus on determinants first, then can extend to analyst outcomes."
    },
])

display(literature_matrix)
save_df(literature_matrix, "31_literature_review_matrix_skeleton.csv")

In [ ]:
# ============================================================
# Cell 22. Final copy-paste summary for ChatGPT
# ============================================================

compact = {
    "research_direction": {
        "title_candidate_1": "Disclosure Corrections, Ownership Structure, and Reporting Quality: Evidence from Korean Open DART Filings",
        "title_candidate_2": "Disclosure Corrections as a Signal of Reporting Quality: Evidence from Korean Open DART Filings",
        "main_focus": "Correction disclosures as reporting quality proxy; governance and ownership determinants",
        "sample_issue_fixed": "Final panel is now forced to equal DART-matched current KOSPI nonfinancial sample when ANALYSIS_SAMPLE_MODE is current_sample.",
        "coverage_issue_fixed": "Coverage uses unique corp_code counts, so it cannot exceed 100%.",
        "year_definition": "fiscal_year is separated from filing_year. Business reports are searched in fiscal_year+1."
    },
    "settings": {
        "analysis_sample_mode": ANALYSIS_SAMPLE_MODE,
        "pilot_mode": PILOT_MODE,
        "start_fiscal_year": START_FISCAL_YEAR,
        "end_fiscal_year": END_FISCAL_YEAR
    },
    "sample_info": {
        "number_of_firms_final_panel": int(final_panel["corp_code"].nunique()),
        "number_of_firm_years_final_panel": int(len(final_panel)),
        "number_of_sectors": int(final_panel["sector"].nunique())
    },
    "sample_selection_final": sample_selection_final.to_dict(orient="records"),
    "coverage_by_year": coverage_by_year.to_dict(orient="records"),
    "current_report_coverage": current_report_coverage.to_dict(orient="records"),
    "yearly_correction_summary": yearly_correction_summary.to_dict(orient="records"),
    "variable_summary": {},
    "missing_rates": {},
    "regression_sample_sizes": regression_sample_sizes.to_dict(orient="records"),
    "regression_key_coefficients": [],
    "paper_style_coefficients": [],
    "material_validation_sample_size": int(len(material_validation_sample)) if "material_validation_sample" in globals() else 0
}

summary_vars = [
    "CorrectionDummy", "CorrectionCount", "LogCorrectionCount",
    "MaterialCorrectionDummy", "MaterialCorrectionCount",
    "OutsideDirectorRatio", "LargestShareholderOwnership",
    "MinorityShareholderCount", "LogMinorityShareholderCount",
    "LogAssets", "Leverage", "ROA", "Loss",
    "Big4Auditor", "CleanOpinion",
    "NonAuditFeeRatio", "AuditFee", "NonAuditFee"
]

for var in summary_vars:
    if var in final_panel.columns:
        s = final_panel[var]
        compact["missing_rates"][var] = float(s.isna().mean())
        
        if pd.api.types.is_numeric_dtype(s):
            compact["variable_summary"][var] = {
                "non_missing": int(s.notna().sum()),
                "mean": None if pd.isna(s.mean()) else float(s.mean()),
                "median": None if pd.isna(s.median()) else float(s.median()),
                "std": None if pd.isna(s.std()) else float(s.std()),
                "min": None if pd.isna(s.min()) else float(s.min()),
                "max": None if pd.isna(s.max()) else float(s.max())
            }

if "coef_table" in globals() and not coef_table.empty:
    important_terms = [
        "OutsideDirectorRatio_w",
        "LargestShareholderOwnership_w",
        "LogMinorityShareholderCount_w",
        "LogAssets_w",
        "Leverage_w",
        "ROA_w",
        "Loss",
        "Big4Auditor",
        "CleanOpinion",
        "NonAuditFeeRatio_w",
        "LogAuditFee_w",
        "LogNonAuditFee_w",
        "OutsideDirectorRatio_w_lag1",
        "LargestShareholderOwnership_w_lag1",
        "LogMinorityShareholderCount_w_lag1",
        "LogAssets_w_lag1",
        "Leverage_w_lag1",
        "ROA_w_lag1",
        "Loss_lag1"
    ]
    
    compact["regression_key_coefficients"] = (
        coef_table[coef_table["variable"].isin(important_terms)]
        .to_dict(orient="records")
    )

if "paper_coef" in globals() and not paper_coef.empty:
    compact["paper_style_coefficients"] = paper_coef.to_dict(orient="records")

print("\n" + "=" * 80)
print("COMPACT COPY-PASTE BLOCK FOR CHATGPT")
print("=" * 80)
print(json.dumps(compact, ensure_ascii=False, indent=2))
print("=" * 80)
print("END OF FINAL RESULT SUMMARY")
print("=" * 80)

with open(OUT_DIR / "copy_paste_summary_for_chatgpt_CLEAN.json", "w", encoding="utf-8") as f:
    json.dump(compact, f, ensure_ascii=False, indent=2)

print("\nSaved files:")
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)